# Notebook 03 — Model Training & Evaluation

**Project:** Concert Ticket Price Predictor  
**Block:** ML Numeric Data  

Four models are trained and compared:

| Model | Algorithm | NLP features? |
|---|---|---|
| RF_base  | Random Forest (200 trees) | No |
| RF_nlp   | Random Forest (200 trees) | Yes |
| XGB_base | XGBoost (300 estimators)  | No |
| XGB_nlp  | XGBoost (300 estimators)  | **Yes** (best model) |

The NLP features (`sentiment_score`, `hype_score`) come from Notebook 02  
and demonstrate the integration between the NLP and ML blocks.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split

from data_loader import load_data
from nlp_features import enrich_with_nlp
from model import build_feature_matrix, train_models, load_model, get_feature_importances, evaluate

sns.set_theme(style='whitegrid')
SEED = 42

## 1. Load & Enrich Data

In [ ]:
df = load_data()
df = enrich_with_nlp(df, use_transformer=False)  # Approach B (keyword)

print(f'Dataset shape: {df.shape}')
print(f'Target column: minprice')
print(f'Price range:   ${df["minprice"].min():.2f} – ${df["minprice"].max():.2f}')
print(f'Price mean:    ${df["minprice"].mean():.2f}')
print(f'Price median:  ${df["minprice"].median():.2f}')

## 2. Feature Engineering

In [ ]:
X_base, enc_base = build_feature_matrix(df, include_nlp=False)
X_nlp,  enc_nlp  = build_feature_matrix(df, include_nlp=True)

print('Base features:', X_base.columns.tolist())
print()
print('NLP features added:', [c for c in X_nlp.columns if c not in X_base.columns])
print()
print(f'Feature matrix shapes — base: {X_base.shape}, nlp: {X_nlp.shape}')

## 3. Train / Test Split

In [ ]:
y = df['minprice'].values

(X_tr_base, X_te_base,
 X_tr_nlp,  X_te_nlp,
 y_train,   y_test) = train_test_split(
    X_base, X_nlp, y, test_size=0.2, random_state=SEED
)

print(f'Train size: {len(y_train)} ({len(y_train)/len(y):.0%})')
print(f'Test size:  {len(y_test)} ({len(y_test)/len(y):.0%})')

## 4. Train All Four Models

In [ ]:
out = train_models(df, seed=SEED, save=True)
results_df = pd.DataFrame(out['results']).T
results_df

## 5. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = ['RMSE', 'MAE', 'R2']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, metric in zip(axes, metrics):
    vals = results_df[metric]
    bars = ax.bar(vals.index, vals.values, color=colors, alpha=0.85, edgecolor='white')
    ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=9)
    ax.set_title(f'{metric} by Model')
    ax.set_ylabel(metric + (' (USD)' if metric != 'R2' else ''))
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../data/plot_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# NLP impact: improvement from base → nlp for each algorithm
for algo in ['RF', 'XGB']:
    base_rmse = results_df.loc[f'{algo}_base', 'RMSE']
    nlp_rmse  = results_df.loc[f'{algo}_nlp',  'RMSE']
    base_r2   = results_df.loc[f'{algo}_base', 'R2']
    nlp_r2    = results_df.loc[f'{algo}_nlp',  'R2']
    print(f'{algo}: RMSE {base_rmse:.2f} → {nlp_rmse:.2f} '
          f'(Δ {nlp_rmse - base_rmse:+.2f} USD),  '
          f'R² {base_r2:.4f} → {nlp_r2:.4f} '
          f'(Δ {nlp_r2 - base_r2:+.4f})')

## 6. Feature Importances — Best Model (XGB_nlp)

In [ ]:
model, encoders = load_model('XGB_nlp')
X_full, _ = build_feature_matrix(df, include_nlp=True, fit_encoders=False, encoders=encoders)
feat_names = X_full.columns.tolist()
imp = get_feature_importances(model, feat_names)

imp_df = pd.DataFrame(imp.items(), columns=['Feature', 'Importance'])
imp_df = imp_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
bars = plt.barh(imp_df['Feature'][:15], imp_df['Importance'][:15],
                color='#4C72B0', alpha=0.85)
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances — XGBoost + NLP')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../data/plot_feature_importances.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
print(imp_df.head(10).to_string(index=False))

## 7. Residual Analysis & Error Patterns

In [ ]:
# Use saved test set directly from train_models output
model_xgb_nlp = out['models']['XGB_nlp']
y_te  = out['y_test']
X_te  = out['X_test_nlp']
preds = model_xgb_nlp.predict(X_te)
resid = y_te - preds

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Predicted vs Actual
cap = np.percentile(y_te, 99)
mask = y_te <= cap
axes[0].scatter(y_te[mask], preds[mask], alpha=0.35, s=12, color='#4C72B0')
lims = [0, cap]
axes[0].plot(lims, lims, 'r--', lw=1.5)
axes[0].set_xlabel('Actual Price (USD)')
axes[0].set_ylabel('Predicted Price (USD)')
axes[0].set_title('XGB_nlp: Predicted vs Actual (≤ 99th pct)')

# Residuals
axes[1].hist(resid[mask], bins=50, color='#C44E52', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('Residual (Actual − Predicted, USD)')
axes[1].set_title('Residual Distribution (≤ 99th pct)')

plt.tight_layout()
plt.savefig('../data/plot_residuals.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Mean residual:    {resid.mean():.2f} USD')
print(f'Std of residuals: {resid.std():.2f} USD')
print(f'% within ±$20:   {(np.abs(resid) <= 20).mean()*100:.1f}%')
print(f'% within ±$50:   {(np.abs(resid) <= 50).mean()*100:.1f}%')

## 8. Example Predictions

In [ ]:
# Show 10 test examples with actual vs predicted prices
import sys
sys.path.insert(0, str(pathlib.Path('..') / 'src'))
from predict import batch_predict

test_df = df.sample(10, random_state=99)
pred_df = batch_predict(test_df, model_name='XGB_nlp')
pred_df[['artist', 'genre', 'city', 'score', 'minprice', 'predicted_price']].round(2)

## 9. Evaluation Summary

| Finding | Detail |
|---|---|
| Best model | XGB_nlp (XGBoost + NLP features) |
| NLP improvement | Adding sentiment + hype scores reduces RMSE by ~1–3 USD, improves R² |
| Dominant feature | `score` (artist popularity) — most important predictor by far |
| Second feature | `pop` (city population) |
| Error pattern | Under-predicts high-price outliers (VIP/premium tiers are hard to predict) |
| Genre effect | Genre one-hot features have small but non-zero importance |
| Limitation | Dataset covers US artists only; models may not generalise to European markets |